In [8]:
import pandas as pd
df = pd.read_csv("BoCo_200.csv")

df = df.dropna(subset=['volunteer_id']) # remove rows where volunteer_id = NaN
df['volunteer_id'] = df['volunteer_id'].astype(int).astype(str).str.zfill(3) # convert volunteer_id into numerical entries following same format

mask = df['Referral'] == 'N'
df.loc[mask, 'ref_name'] = 'N/A'

# reformating of csv file entries
for i in range(1, 3):
    event = f"event_date.{i}"
    sign_up = f"sign_up.{i}"
    attend = f"attended.{i}"
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"

# formating date entries and naming days of the week for future use
    dt = pd.to_datetime(df[event])
    df[event] = pd.to_datetime(df[event])
    df[event] = dt.dt.strftime("%m/%d/%Y")

    df[f"day_of_week.{i}"] = dt.dt.day_name()

# cleaning up attendance entries
    mask = (df[sign_up] == 'N') & (df[attend].isna())
    df.loc[mask, attend] = 'N/A'

# formating time
for i in range(1, 3):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"

    dt_in = pd.to_datetime(df[time_in], format="%I:%M %p", errors='coerce')
    dt_out = pd.to_datetime(df[time_out], format="%I:%M %p", errors='coerce')
    df[time_in] = dt_in.dt.strftime("%I:%M %p")
    df[time_out] = dt_out.dt.strftime("%I:%M %p")

for i in range(1, 3):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"
    hours = f"hours.{i}"

    df[hours] = (
        pd.to_datetime(df[time_out], format="%I:%M %p", errors='coerce') - pd.to_datetime(df[time_in], format="%I:%M %p", errors='coerce')
    ).dt.total_seconds() / 3600     # seconds -> hrs

    df[hours] = df[hours].fillna(0)

df['total_hours'] = df['hours.1'] + df['hours.2']

mask = (df['total_hours'].isna())
df.loc[mask, 'total_hours'] = 0.0

# cleaning up empty time entries
for i in range(1, 3):
    time_in = f"time_in.{i}"
    time_out = f"time_out.{i}"

    mask = (df[time_in].isna())
    df.loc[mask, [time_in]] = 'N/A'

    mask = (df[time_out].isna())
    df.loc[mask, [time_out]] = 'N/A'

# no case match
df['ref_name'] = df['ref_name'].str.title()

# reorder to keep everything grouped accordingly
new_order = [
    'volunteer_id', 'name_first', 'name_last', 'email', 'phone',
    'Referral', 'ref_name',
    'event_date.1', 'sign_up.1', 'attended.1', 'time_in.1', 'time_out.1', 'day_of_week.1', 'hours.1',
    'event_date.2', 'sign_up.2', 'attended.2', 'time_in.2', 'time_out.2', 'day_of_week.2', 'hours.2',
    'total_hours'
]

df = df[new_order]